# FAISS Project – SIFT Starter Experiment

This notebook loads the SIFT dataset, builds exact ground truth, and runs a first IVFPQ experiment with Recall@10 and QPS.

In [2]:
import sys
from pathlib import Path

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [4]:
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

C:\Users\vaspn\Downloads\Vector-Search-FAISS


In [5]:
from src.data_loading import load_sift
from src.benchmarks import build_ground_truth, benchmark_index
from src.metrics import recall_at_k

## Load SIFT

In [ ]:
xb, xq, xlearn, gt = load_sift()

print("xb:", xb.shape, xb.dtype)
print("xq:", xq.shape, xq.dtype)
print("xlearn:", xlearn.shape, xlearn.dtype)
print("gt:", gt.shape, gt.dtype)

## Exact Ground Truth

In [ ]:
k = 10
d = xb.shape[1]
metric = faiss.METRIC_L2

D_gt, I_gt = build_ground_truth(xb, xq, k, metric)
print("D_gt shape:", D_gt.shape)
print("I_gt shape:", I_gt.shape)

## First IVFPQ Experiment

In [ ]:
factory = "IVF4096,PQ32x8"
index = faiss.index_factory(d, factory, metric)

train_data = xlearn if xlearn is not None and len(xlearn) > 0 else xb
if not index.is_trained:
    index.train(train_data)

index.add(xb)

In [ ]:
ps = faiss.ParameterSpace()
ps.set_index_parameter(index, "nprobe", 16)

D, I, total_time, qps = benchmark_index(index, xq, k)
recall = recall_at_k(I, I_gt, k)

print("Factory:", factory)
print("Recall@10:", recall)
print("QPS:", qps)
print("Total search time:", total_time)

## Notes

This is the first working benchmark. Next steps are parameter sweeps for IVFPQ, HNSWSQ, and LSH.

## IVFPQ Sweep on SIFT

In [ ]:
from src.indexes import set_ivfpq_nprobe, set_hnswsq_efsearch

ivfpq_factories = ["IVF4096,PQ8x8", "IVF4096,PQ16x8", "IVF4096,PQ32x8"]
nprobe_values = [1, 4, 8, 16, 32, 64]

In [ ]:
ivfpq_rows = []
train_data = xlearn if xlearn is not None and len(xlearn) > 0 else xb

for factory in ivfpq_factories:
    index = faiss.index_factory(d, factory, metric)
    if not index.is_trained:
        index.train(train_data)
    index.add(xb)

    for nprobe in nprobe_values:
        set_ivfpq_nprobe(index, int(nprobe))
        D, I, total_time, qps = benchmark_index(index, xq, k)
        recall = recall_at_k(I, I_gt, k)

        ivfpq_rows.append({
            "dataset": "sift",
            "method": "IVFPQ",
            "factory": factory,
            "search_param_name": "nprobe",
            "search_param_value": nprobe,
            "recall_at_10": recall,
            "qps": qps,
            "total_time_sec": total_time,
        })

In [ ]:
ivfpq_results_df = pd.DataFrame(ivfpq_rows)
ivfpq_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
ivfpq_path = tables_dir / "sift_ivfpq_results.csv"
ivfpq_results_df.to_csv(ivfpq_path, index=False)
print(f"Saved: {ivfpq_path}")

## HNSWSQ Sweep on SIFT

In [ ]:
hnsw_factories = ["HNSW32_SQ4", "HNSW32_SQ6", "HNSW32_SQ8"]
efsearch_values = [16, 32, 64, 128, 256]

In [ ]:
hnswsq_rows = []
train_data = xlearn if xlearn is not None and len(xlearn) > 0 else xb

for factory in hnsw_factories:
    index = faiss.index_factory(d, factory, metric)
    if not index.is_trained:
        index.train(train_data)
    index.add(xb)

    for efsearch in efsearch_values:
        set_hnswsq_efsearch(index, int(efsearch))
        D, I, total_time, qps = benchmark_index(index, xq, k)
        recall = recall_at_k(I, I_gt, k)

        hnswsq_rows.append({
            "dataset": "sift",
            "method": "HNSWSQ",
            "factory": factory,
            "search_param_name": "efSearch",
            "search_param_value": efsearch,
            "recall_at_10": recall,
            "qps": qps,
            "total_time_sec": total_time,
        })

In [ ]:
hnswsq_results_df = pd.DataFrame(hnswsq_rows)
hnswsq_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
hnswsq_path = tables_dir / "sift_hnswsq_results.csv"
hnswsq_results_df.to_csv(hnswsq_path, index=False)
print(f"Saved: {hnswsq_path}")

## LSH Sweep on SIFT

In [ ]:
lsh_bits = [128, 256, 512]

In [ ]:
lsh_rows = []

for nbits in lsh_bits:
    index = faiss.IndexLSH(d, int(nbits))
    index.add(xb)

    D, I, total_time, qps = benchmark_index(index, xq, k)
    recall = recall_at_k(I, I_gt, k)

    lsh_rows.append({
        "dataset": "sift",
        "method": "LSH",
        "factory": "IndexLSH",
        "search_param_name": "nbits",
        "search_param_value": nbits,
        "recall_at_10": recall,
        "qps": qps,
        "total_time_sec": total_time,
    })

In [ ]:
lsh_results_df = pd.DataFrame(lsh_rows)
lsh_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
lsh_path = tables_dir / "sift_lsh_results.csv"
lsh_results_df.to_csv(lsh_path, index=False)
print(f"Saved: {lsh_path}")

## Combined SIFT Results

In [ ]:
sift_results_df = pd.concat([ivfpq_results_df, hnswsq_results_df, lsh_results_df], ignore_index=True)
sift_results_df.shape

In [ ]:
sift_results_sorted_df = sift_results_df.sort_values(["recall_at_10", "qps"], ascending=[False, False]).reset_index(drop=True)
sift_results_sorted_df

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
all_path = tables_dir / "sift_all_results.csv"
sift_results_df.to_csv(all_path, index=False)
print(f"Saved: {all_path}")

## SIFT Recall–QPS Plots

In [ ]:
plt.figure(figsize=(8, 5))

for method in ["IVFPQ", "HNSWSQ", "LSH"]:
    subset = sift_results_df[sift_results_df["method"] == method]
    plt.scatter(subset["qps"], subset["recall_at_10"], label=method, alpha=0.8)

plt.xlabel("QPS")
plt.ylabel("Recall@10")
plt.title("SIFT: Recall@10 vs QPS")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
plot_path = plots_dir / "sift_recall_vs_qps.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {plot_path}")

## Best SIFT Configurations

In [ ]:
top_recall_df = sift_results_df.sort_values(["recall_at_10", "qps"], ascending=[False, False]).head(5).copy()
top_recall_df["summary"] = "top_recall"

top_qps_df = sift_results_df.sort_values(["qps", "recall_at_10"], ascending=[False, False]).head(5).copy()
top_qps_df["summary"] = "top_qps"

best_sift_summary_df = pd.concat([top_recall_df, top_qps_df], ignore_index=True)[
    ["summary", "method", "factory", "search_param_name", "search_param_value", "recall_at_10", "qps", "total_time_sec"]
]
best_sift_summary_df

## SIFT Experiment Notes

- Next step: RC analysis.
- Next step: hard/easy query groups.
- Next step: noisy query robustness evaluation.
- Next step: repeat the same scaffold for GIST and then GloVe.

## Relative Contrast (RC) on SIFT

In [ ]:
from src.rc_analysis import compute_rc, add_noise

rc_values = compute_rc(xb, xq, k, metric)

In [ ]:
rc_df = pd.DataFrame({
    "query_id": np.arange(len(rc_values), dtype=np.int64),
    "rc": rc_values,
})
rc_df.head()

In [ ]:
rc_stats_df = pd.DataFrame([
    {
        "mean": float(rc_df["rc"].mean()),
        "std": float(rc_df["rc"].std()),
        "min": float(rc_df["rc"].min()),
        "max": float(rc_df["rc"].max()),
    }
])
rc_quantiles = rc_df["rc"].quantile([0.25, 0.5, 0.75]).rename("rc_quantile")

print("RC basic stats:")
display(rc_stats_df)
print("RC quantiles (0.25, 0.5, 0.75):")
display(rc_quantiles)

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
rc_path = tables_dir / "sift_rc_values.csv"
rc_df.to_csv(rc_path, index=False)
print(f"Saved: {rc_path}")

## Easy vs Hard Queries

In [ ]:
q25 = float(rc_df["rc"].quantile(0.25))
q75 = float(rc_df["rc"].quantile(0.75))

hard_query_idx = np.where(rc_values <= q25)[0]
easy_query_idx = np.where(rc_values >= q75)[0]

In [ ]:
print(f"Hard queries: {len(hard_query_idx)}")
print(f"Easy queries: {len(easy_query_idx)}")

In [ ]:
rc_split_summary_df = pd.DataFrame([
    {
        "dataset": "sift",
        "q25": q25,
        "q75": q75,
        "hard_query_count": int(len(hard_query_idx)),
        "easy_query_count": int(len(easy_query_idx)),
        "total_query_count": int(len(rc_values)),
    }
])

tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
rc_split_path = tables_dir / "sift_rc_split_summary.csv"
rc_split_summary_df.to_csv(rc_split_path, index=False)
print(f"Saved: {rc_split_path}")
rc_split_summary_df

## Recall on Easy vs Hard Queries

In [ ]:
representative_configs = [
    {"method": "IVFPQ", "factory": "IVF4096,PQ32x8", "setting_name": "nprobe", "setting_value": 64},
    {"method": "HNSWSQ", "factory": "HNSW32_SQ8", "setting_name": "efSearch", "setting_value": 256},
    {"method": "LSH", "factory": "IndexLSH", "setting_name": "nbits", "setting_value": 512},
]

easy_hard_rows = []
train_data = xlearn if xlearn is not None and len(xlearn) > 0 else xb

for cfg in representative_configs:
    method = cfg["method"]
    factory = cfg["factory"]
    setting_name = cfg["setting_name"]
    setting_value = cfg["setting_value"]

    if method == "LSH":
        index = faiss.IndexLSH(d, int(setting_value))
    else:
        index = faiss.index_factory(d, factory, metric)
        if not index.is_trained:
            index.train(train_data)
    index.add(xb)

    ps = faiss.ParameterSpace()
    if method == "IVFPQ":
        ps.set_index_parameter(index, "nprobe", int(setting_value))
    elif method == "HNSWSQ":
        ps.set_index_parameter(index, "efSearch", int(setting_value))

    D, I, total_time, qps = benchmark_index(index, xq, k)
    recall_all = recall_at_k(I, I_gt, k)
    recall_hard = recall_at_k(I[hard_query_idx], I_gt[hard_query_idx], k)
    recall_easy = recall_at_k(I[easy_query_idx], I_gt[easy_query_idx], k)

    easy_hard_rows.append({
        "dataset": "sift",
        "method": method,
        "factory": factory,
        "setting_name": setting_name,
        "setting_value": setting_value,
        "recall_all": recall_all,
        "recall_hard": recall_hard,
        "recall_easy": recall_easy,
        "qps": qps,
    })

sift_easy_hard_df = pd.DataFrame(easy_hard_rows)

tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
easy_hard_path = tables_dir / "sift_easy_hard_results.csv"
sift_easy_hard_df.to_csv(easy_hard_path, index=False)
print(f"Saved: {easy_hard_path}")
sift_easy_hard_df

## RC Plots

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8, 5))
plt.hist(rc_df["rc"], bins=40, alpha=0.85, edgecolor="black")
plt.xlabel("RC")
plt.ylabel("Count")
plt.title("SIFT RC Distribution")
plt.grid(alpha=0.3)
plt.tight_layout()

rc_hist_path = plots_dir / "sift_rc_histogram.png"
plt.savefig(rc_hist_path, dpi=150, bbox_inches="tight")
print(f"Saved: {rc_hist_path}")

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

methods = sift_easy_hard_df["method"].tolist()
x = np.arange(len(methods))
width = 0.25

plt.figure(figsize=(9, 5))
plt.bar(x - width, sift_easy_hard_df["recall_all"], width=width, label="recall_all")
plt.bar(x, sift_easy_hard_df["recall_hard"], width=width, label="recall_hard")
plt.bar(x + width, sift_easy_hard_df["recall_easy"], width=width, label="recall_easy")
plt.xticks(x, methods)
plt.ylabel("Recall@10")
plt.title("SIFT: Recall on All vs Hard vs Easy Queries")
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()

easy_hard_plot_path = plots_dir / "sift_easy_vs_hard_recall.png"
plt.savefig(easy_hard_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {easy_hard_plot_path}")

## Noisy Query Robustness on SIFT

In [ ]:
sigma_values = [0.00, 0.01, 0.05, 0.10]

In [ ]:
from src.preprocessing import l2_normalize

noise_rows = []
train_data = xlearn if xlearn is not None and len(xlearn) > 0 else xb

for cfg in representative_configs:
    method = cfg["method"]
    factory = cfg["factory"]
    setting_value = cfg["setting_value"]

    if method == "LSH":
        index = faiss.IndexLSH(d, int(setting_value))
    else:
        index = faiss.index_factory(d, factory, metric)
        if not index.is_trained:
            index.train(train_data)
    index.add(xb)

    ps = faiss.ParameterSpace()
    if method == "IVFPQ":
        ps.set_index_parameter(index, "nprobe", int(setting_value))
    elif method == "HNSWSQ":
        ps.set_index_parameter(index, "efSearch", int(setting_value))

    for sigma in sigma_values:
        xq_noisy = add_noise(xq, sigma)
        if metric == faiss.METRIC_INNER_PRODUCT:
            xq_noisy = l2_normalize(xq_noisy)

        D, I, total_time, qps = benchmark_index(index, xq_noisy, k)
        recall = recall_at_k(I, I_gt, k)

        noise_rows.append({
            "dataset": "sift",
            "method": method,
            "factory": factory,
            "noise_sigma": float(sigma),
            "recall_at_10": recall,
            "qps": qps,
        })

sift_noise_df = pd.DataFrame(noise_rows)

tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
noise_path = tables_dir / "sift_noisy_query_results.csv"
sift_noise_df.to_csv(noise_path, index=False)
print(f"Saved: {noise_path}")
sift_noise_df

## Noisy Query Plots

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8, 5))
for method in ["IVFPQ", "HNSWSQ", "LSH"]:
    subset = sift_noise_df[sift_noise_df["method"] == method].sort_values("noise_sigma")
    plt.plot(subset["noise_sigma"], subset["recall_at_10"], marker="o", label=method)

plt.xlabel("Noise sigma")
plt.ylabel("Recall@10")
plt.title("SIFT: Recall@10 vs Query Noise")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

noise_plot_path = plots_dir / "sift_noise_vs_recall.png"
plt.savefig(noise_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {noise_plot_path}")

## SIFT RC and Noise Notes

Next step is to repeat the same experiment scaffold for GIST, and later implement GloVe with normalization + inner product.

# GIST Dataset Experiments

In [ ]:
from src.data_loading import load_gist

xbg, xqg, xlearng, gtg = load_gist()

print("xbg:", xbg.shape, xbg.dtype)
print("xqg:", xqg.shape, xqg.dtype)
print("xlearng:", xlearng.shape, xlearng.dtype)
print("gtg:", gtg.shape, gtg.dtype)

In [ ]:
kg = 10
dg = xbg.shape[1]
metric_g = faiss.METRIC_L2

D_gt_g, I_gt_g = build_ground_truth(xbg, xqg, kg, metric_g)
print("D_gt_g:", D_gt_g.shape)
print("I_gt_g:", I_gt_g.shape)

## GIST IVFPQ Sweep

In [ ]:
from src.indexes import set_ivfpq_nprobe, set_hnswsq_efsearch

gist_ivfpq_factories = ["IVF4096,PQ8x8", "IVF4096,PQ16x8", "IVF4096,PQ32x8"]
gist_nprobe_values = [1, 4, 8, 16, 32, 64]

In [ ]:
gist_ivfpq_rows = []
train_data_g = xlearng if xlearng is not None and len(xlearng) > 0 else xbg

for factory in gist_ivfpq_factories:
    index = faiss.index_factory(dg, factory, metric_g)
    if not index.is_trained:
        index.train(train_data_g)
    index.add(xbg)

    for nprobe in gist_nprobe_values:
        set_ivfpq_nprobe(index, int(nprobe))
        D, I, total_time, qps = benchmark_index(index, xqg, kg)
        recall = recall_at_k(I, I_gt_g, kg)

        gist_ivfpq_rows.append({
            "dataset": "gist",
            "method": "IVFPQ",
            "factory": factory,
            "search_param_name": "nprobe",
            "search_param_value": nprobe,
            "recall_at_10": recall,
            "qps": qps,
            "total_time_sec": total_time,
        })

In [ ]:
gist_ivfpq_results_df = pd.DataFrame(gist_ivfpq_rows)
gist_ivfpq_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
gist_ivfpq_path = tables_dir / "gist_ivfpq_results.csv"
gist_ivfpq_results_df.to_csv(gist_ivfpq_path, index=False)
print(f"Saved: {gist_ivfpq_path}")

## GIST HNSWSQ Sweep

In [ ]:
gist_hnsw_factories = ["HNSW32_SQ4", "HNSW32_SQ6", "HNSW32_SQ8"]
gist_efsearch_values = [16, 32, 64, 128, 256]

In [ ]:
gist_hnswsq_rows = []
train_data_g = xlearng if xlearng is not None and len(xlearng) > 0 else xbg

for factory in gist_hnsw_factories:
    index = faiss.index_factory(dg, factory, metric_g)
    if not index.is_trained:
        index.train(train_data_g)
    index.add(xbg)

    for efsearch in gist_efsearch_values:
        set_hnswsq_efsearch(index, int(efsearch))
        D, I, total_time, qps = benchmark_index(index, xqg, kg)
        recall = recall_at_k(I, I_gt_g, kg)

        gist_hnswsq_rows.append({
            "dataset": "gist",
            "method": "HNSWSQ",
            "factory": factory,
            "search_param_name": "efSearch",
            "search_param_value": efsearch,
            "recall_at_10": recall,
            "qps": qps,
            "total_time_sec": total_time,
        })

In [ ]:
gist_hnswsq_results_df = pd.DataFrame(gist_hnswsq_rows)
gist_hnswsq_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
gist_hnswsq_path = tables_dir / "gist_hnswsq_results.csv"
gist_hnswsq_results_df.to_csv(gist_hnswsq_path, index=False)
print(f"Saved: {gist_hnswsq_path}")

## GIST LSH Sweep

In [ ]:
gist_lsh_bits = [128, 256, 512]

In [ ]:
gist_lsh_rows = []

for nbits in gist_lsh_bits:
    index = faiss.IndexLSH(dg, int(nbits))
    index.add(xbg)

    D, I, total_time, qps = benchmark_index(index, xqg, kg)
    recall = recall_at_k(I, I_gt_g, kg)

    gist_lsh_rows.append({
        "dataset": "gist",
        "method": "LSH",
        "factory": "IndexLSH",
        "search_param_name": "nbits",
        "search_param_value": nbits,
        "recall_at_10": recall,
        "qps": qps,
        "total_time_sec": total_time,
    })

In [ ]:
gist_lsh_results_df = pd.DataFrame(gist_lsh_rows)
gist_lsh_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
gist_lsh_path = tables_dir / "gist_lsh_results.csv"
gist_lsh_results_df.to_csv(gist_lsh_path, index=False)
print(f"Saved: {gist_lsh_path}")

## Combined GIST Results

In [ ]:
gist_results_df = pd.concat([gist_ivfpq_results_df, gist_hnswsq_results_df, gist_lsh_results_df], ignore_index=True)
gist_results_df.shape

In [ ]:
gist_results_sorted_df = gist_results_df.sort_values(["recall_at_10", "qps"], ascending=[False, False]).reset_index(drop=True)
gist_results_sorted_df

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
gist_all_path = tables_dir / "gist_all_results.csv"
gist_results_df.to_csv(gist_all_path, index=False)
print(f"Saved: {gist_all_path}")

## GIST Recall?QPS Plot

In [ ]:
plt.figure(figsize=(8, 5))

for method in ["IVFPQ", "HNSWSQ", "LSH"]:
    subset = gist_results_df[gist_results_df["method"] == method]
    plt.scatter(subset["qps"], subset["recall_at_10"], label=method, alpha=0.8)

plt.xlabel("QPS")
plt.ylabel("Recall@10")
plt.title("GIST: Recall@10 vs QPS")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
gist_recall_qps_path = plots_dir / "gist_recall_vs_qps.png"
plt.savefig(gist_recall_qps_path, dpi=150, bbox_inches="tight")
print(f"Saved: {gist_recall_qps_path}")

## GIST RC Analysis

In [ ]:
gist_rc_values = compute_rc(xbg, xqg, kg, metric_g)

In [ ]:
gist_rc_df = pd.DataFrame({
    "query_id": np.arange(len(gist_rc_values), dtype=np.int64),
    "rc": gist_rc_values,
})
gist_rc_df.head()

In [ ]:
gist_rc_stats_df = pd.DataFrame([
    {
        "mean": float(gist_rc_df["rc"].mean()),
        "std": float(gist_rc_df["rc"].std()),
        "min": float(gist_rc_df["rc"].min()),
        "max": float(gist_rc_df["rc"].max()),
    }
])
gist_rc_quantiles = gist_rc_df["rc"].quantile([0.25, 0.5, 0.75]).rename("rc_quantile")

print("GIST RC basic stats:")
display(gist_rc_stats_df)
print("GIST RC quantiles (0.25, 0.5, 0.75):")
display(gist_rc_quantiles)

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
gist_rc_path = tables_dir / "gist_rc_values.csv"
gist_rc_df.to_csv(gist_rc_path, index=False)
print(f"Saved: {gist_rc_path}")

## GIST Easy vs Hard Queries

In [ ]:
gist_q25 = float(gist_rc_df["rc"].quantile(0.25))
gist_q75 = float(gist_rc_df["rc"].quantile(0.75))

gist_hard_query_idx = np.where(gist_rc_values <= gist_q25)[0]
gist_easy_query_idx = np.where(gist_rc_values >= gist_q75)[0]

In [ ]:
print(f"GIST hard queries: {len(gist_hard_query_idx)}")
print(f"GIST easy queries: {len(gist_easy_query_idx)}")

In [ ]:
gist_rc_split_summary_df = pd.DataFrame([
    {
        "dataset": "gist",
        "q25": gist_q25,
        "q75": gist_q75,
        "hard_query_count": int(len(gist_hard_query_idx)),
        "easy_query_count": int(len(gist_easy_query_idx)),
        "total_query_count": int(len(gist_rc_values)),
    }
])

tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
gist_rc_split_path = tables_dir / "gist_rc_split_summary.csv"
gist_rc_split_summary_df.to_csv(gist_rc_split_path, index=False)
print(f"Saved: {gist_rc_split_path}")
gist_rc_split_summary_df

## GIST Recall on Easy vs Hard Queries

In [ ]:
gist_representative_configs = [
    {"method": "IVFPQ", "factory": "IVF4096,PQ32x8", "setting_name": "nprobe", "setting_value": 64},
    {"method": "HNSWSQ", "factory": "HNSW32_SQ8", "setting_name": "efSearch", "setting_value": 256},
    {"method": "LSH", "factory": "IndexLSH", "setting_name": "nbits", "setting_value": 512},
]

gist_easy_hard_rows = []
train_data_g = xlearng if xlearng is not None and len(xlearng) > 0 else xbg

for cfg in gist_representative_configs:
    method = cfg["method"]
    factory = cfg["factory"]
    setting_name = cfg["setting_name"]
    setting_value = cfg["setting_value"]

    if method == "LSH":
        index = faiss.IndexLSH(dg, int(setting_value))
    else:
        index = faiss.index_factory(dg, factory, metric_g)
        if not index.is_trained:
            index.train(train_data_g)
    index.add(xbg)

    if method == "IVFPQ":
        set_ivfpq_nprobe(index, int(setting_value))
    elif method == "HNSWSQ":
        set_hnswsq_efsearch(index, int(setting_value))

    D, I, total_time, qps = benchmark_index(index, xqg, kg)
    recall_all = recall_at_k(I, I_gt_g, kg)
    recall_hard = recall_at_k(I[gist_hard_query_idx], I_gt_g[gist_hard_query_idx], kg)
    recall_easy = recall_at_k(I[gist_easy_query_idx], I_gt_g[gist_easy_query_idx], kg)

    gist_easy_hard_rows.append({
        "dataset": "gist",
        "method": method,
        "factory": factory,
        "setting_name": setting_name,
        "setting_value": setting_value,
        "recall_all": recall_all,
        "recall_hard": recall_hard,
        "recall_easy": recall_easy,
        "qps": qps,
    })

gist_easy_hard_df = pd.DataFrame(gist_easy_hard_rows)

tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
gist_easy_hard_path = tables_dir / "gist_easy_hard_results.csv"
gist_easy_hard_df.to_csv(gist_easy_hard_path, index=False)
print(f"Saved: {gist_easy_hard_path}")
gist_easy_hard_df

## GIST RC Plots

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8, 5))
plt.hist(gist_rc_df["rc"], bins=40, alpha=0.85, edgecolor="black")
plt.xlabel("RC")
plt.ylabel("Count")
plt.title("GIST RC Distribution")
plt.grid(alpha=0.3)
plt.tight_layout()

gist_rc_hist_path = plots_dir / "gist_rc_histogram.png"
plt.savefig(gist_rc_hist_path, dpi=150, bbox_inches="tight")
print(f"Saved: {gist_rc_hist_path}")

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

methods = gist_easy_hard_df["method"].tolist()
x = np.arange(len(methods))
width = 0.25

plt.figure(figsize=(9, 5))
plt.bar(x - width, gist_easy_hard_df["recall_all"], width=width, label="recall_all")
plt.bar(x, gist_easy_hard_df["recall_hard"], width=width, label="recall_hard")
plt.bar(x + width, gist_easy_hard_df["recall_easy"], width=width, label="recall_easy")
plt.xticks(x, methods)
plt.ylabel("Recall@10")
plt.title("GIST: Recall on All vs Hard vs Easy Queries")
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()

gist_easy_hard_plot_path = plots_dir / "gist_easy_vs_hard_recall.png"
plt.savefig(gist_easy_hard_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {gist_easy_hard_plot_path}")

## GIST Noisy Query Robustness

In [ ]:
gist_sigma_values = [0.00, 0.01, 0.05, 0.10]

In [ ]:
gist_noise_rows = []
train_data_g = xlearng if xlearng is not None and len(xlearng) > 0 else xbg

for cfg in gist_representative_configs:
    method = cfg["method"]
    factory = cfg["factory"]
    setting_value = cfg["setting_value"]

    if method == "LSH":
        index = faiss.IndexLSH(dg, int(setting_value))
    else:
        index = faiss.index_factory(dg, factory, metric_g)
        if not index.is_trained:
            index.train(train_data_g)
    index.add(xbg)

    if method == "IVFPQ":
        set_ivfpq_nprobe(index, int(setting_value))
    elif method == "HNSWSQ":
        set_hnswsq_efsearch(index, int(setting_value))

    for sigma in gist_sigma_values:
        xqg_noisy = add_noise(xqg, sigma)
        D, I, total_time, qps = benchmark_index(index, xqg_noisy, kg)
        recall = recall_at_k(I, I_gt_g, kg)

        gist_noise_rows.append({
            "dataset": "gist",
            "method": method,
            "factory": factory,
            "noise_sigma": float(sigma),
            "recall_at_10": recall,
            "qps": qps,
        })

gist_noise_df = pd.DataFrame(gist_noise_rows)

tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
gist_noise_path = tables_dir / "gist_noisy_query_results.csv"
gist_noise_df.to_csv(gist_noise_path, index=False)
print(f"Saved: {gist_noise_path}")
gist_noise_df

## GIST Noisy Query Plot

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8, 5))
for method in ["IVFPQ", "HNSWSQ", "LSH"]:
    subset = gist_noise_df[gist_noise_df["method"] == method].sort_values("noise_sigma")
    plt.plot(subset["noise_sigma"], subset["recall_at_10"], marker="o", label=method)

plt.xlabel("Noise sigma")
plt.ylabel("Recall@10")
plt.title("GIST: Recall@10 vs Query Noise")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

gist_noise_plot_path = plots_dir / "gist_noise_vs_recall.png"
plt.savefig(gist_noise_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {gist_noise_plot_path}")

## GIST Notes

Next step is implementing the GloVe dataset pipeline with 100d vectors, L2 normalization, and inner product search.

# GloVe Dataset Experiments

We use `data/raw/glove/glove.twitter.27B.100d.txt` as the source embeddings.

This dataset is treated as angular, so vectors are L2-normalized and search uses inner product.

In [ ]:
glove_txt_path = PROJECT_ROOT / "data" / "raw" / "glove" / "glove.twitter.27B.100d.txt"
print(glove_txt_path)

In [ ]:
max_vectors = 200000
glove_rows = []

with glove_txt_path.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        first_space = line.find(" ")
        if first_space == -1:
            continue
        vec = np.fromstring(line[first_space + 1 :], sep=" ", dtype=np.float32)
        if vec.size != 100:
            continue
        glove_rows.append(vec)
        if len(glove_rows) >= max_vectors:
            break

if not glove_rows:
    raise ValueError(f"No valid GloVe vectors parsed from: {glove_txt_path}")

glove_vectors = np.vstack(glove_rows).astype(np.float32, copy=False)
print("glove_vectors:", glove_vectors.shape, glove_vectors.dtype)

In [ ]:
seed_glove = 42
rng_glove = np.random.default_rng(seed_glove)

n_query = min(10000, max(1000, len(glove_vectors) // 20))
if n_query >= len(glove_vectors):
    raise ValueError("Not enough vectors to create non-overlapping base/query split.")

perm_glove = rng_glove.permutation(len(glove_vectors))
query_idx_glove = perm_glove[:n_query]
base_idx_glove = perm_glove[n_query:]

xb_glove = glove_vectors[base_idx_glove].astype(np.float32, copy=False)
xq_glove = glove_vectors[query_idx_glove].astype(np.float32, copy=False)

overlap_count_glove = np.intersect1d(base_idx_glove, query_idx_glove).size
print("xb_glove:", xb_glove.shape, xb_glove.dtype)
print("xq_glove:", xq_glove.shape, xq_glove.dtype)
print("overlap_count:", int(overlap_count_glove))

In [ ]:
from src.preprocessing import l2_normalize

xb_glove = l2_normalize(xb_glove)
xq_glove = l2_normalize(xq_glove)

In [ ]:
k_glove = 10
d_glove = xb_glove.shape[1]
metric_glove = faiss.METRIC_INNER_PRODUCT

In [ ]:
D_gt_glove, I_gt_glove = build_ground_truth(xb_glove, xq_glove, k_glove, metric_glove)
print("D_gt_glove:", D_gt_glove.shape)
print("I_gt_glove:", I_gt_glove.shape)

## GloVe IVFPQ Sweep

In [ ]:
glove_ivfpq_factories = ["IVF4096,PQ10x8", "IVF4096,PQ20x8", "IVF4096,PQ25x8"]
glove_nprobe_values = [1, 4, 8, 16, 32, 64]

glove_valid_m_values = [int(f.split(",PQ")[1].split("x")[0]) for f in glove_ivfpq_factories]
assert all(d_glove % m == 0 for m in glove_valid_m_values), "Invalid PQ subquantizer count for d_glove"
print("Using GloVe IVFPQ factories:", glove_ivfpq_factories)


In [ ]:
glove_ivfpq_rows = []
train_data_glove = xb_glove

for factory in glove_ivfpq_factories:
    index = faiss.index_factory(d_glove, factory, metric_glove)
    if not index.is_trained:
        index.train(train_data_glove)
    index.add(xb_glove)

    for nprobe in glove_nprobe_values:
        set_ivfpq_nprobe(index, int(nprobe))
        D, I, total_time, qps = benchmark_index(index, xq_glove, k_glove)
        recall = recall_at_k(I, I_gt_glove, k_glove)

        glove_ivfpq_rows.append({
            "dataset": "glove",
            "method": "IVFPQ",
            "factory": factory,
            "search_param_name": "nprobe",
            "search_param_value": nprobe,
            "recall_at_10": recall,
            "qps": qps,
            "total_time_sec": total_time,
        })

In [ ]:
glove_ivfpq_results_df = pd.DataFrame(glove_ivfpq_rows)
glove_ivfpq_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
glove_ivfpq_path = tables_dir / "glove_ivfpq_results.csv"
glove_ivfpq_results_df.to_csv(glove_ivfpq_path, index=False)
print(f"Saved: {glove_ivfpq_path}")

## GloVe HNSWSQ Sweep

In [ ]:
glove_hnsw_factories = ["HNSW32_SQ4", "HNSW32_SQ6", "HNSW32_SQ8"]
glove_efsearch_values = [16, 32, 64, 128, 256]

In [ ]:
glove_hnswsq_rows = []
train_data_glove = xb_glove

for factory in glove_hnsw_factories:
    index = faiss.index_factory(d_glove, factory, metric_glove)
    if not index.is_trained:
        index.train(train_data_glove)
    index.add(xb_glove)

    for efsearch in glove_efsearch_values:
        set_hnswsq_efsearch(index, int(efsearch))
        D, I, total_time, qps = benchmark_index(index, xq_glove, k_glove)
        recall = recall_at_k(I, I_gt_glove, k_glove)

        glove_hnswsq_rows.append({
            "dataset": "glove",
            "method": "HNSWSQ",
            "factory": factory,
            "search_param_name": "efSearch",
            "search_param_value": efsearch,
            "recall_at_10": recall,
            "qps": qps,
            "total_time_sec": total_time,
        })

In [ ]:
glove_hnswsq_results_df = pd.DataFrame(glove_hnswsq_rows)
glove_hnswsq_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
glove_hnswsq_path = tables_dir / "glove_hnswsq_results.csv"
glove_hnswsq_results_df.to_csv(glove_hnswsq_path, index=False)
print(f"Saved: {glove_hnswsq_path}")

## GloVe LSH Sweep

In [ ]:
glove_lsh_bits = [128, 256, 512]

In [ ]:
glove_lsh_rows = []

for nbits in glove_lsh_bits:
    index = faiss.IndexLSH(d_glove, int(nbits))
    index.add(xb_glove)

    D, I, total_time, qps = benchmark_index(index, xq_glove, k_glove)
    recall = recall_at_k(I, I_gt_glove, k_glove)

    glove_lsh_rows.append({
        "dataset": "glove",
        "method": "LSH",
        "factory": "IndexLSH",
        "search_param_name": "nbits",
        "search_param_value": nbits,
        "recall_at_10": recall,
        "qps": qps,
        "total_time_sec": total_time,
    })

In [ ]:
glove_lsh_results_df = pd.DataFrame(glove_lsh_rows)
glove_lsh_results_df.head()

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
glove_lsh_path = tables_dir / "glove_lsh_results.csv"
glove_lsh_results_df.to_csv(glove_lsh_path, index=False)
print(f"Saved: {glove_lsh_path}")

In [ ]:
glove_results_df = pd.concat([glove_ivfpq_results_df, glove_hnswsq_results_df, glove_lsh_results_df], ignore_index=True)
glove_results_df.shape

In [ ]:
glove_results_sorted_df = glove_results_df.sort_values(["recall_at_10", "qps"], ascending=[False, False]).reset_index(drop=True)
glove_results_sorted_df

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
glove_all_path = tables_dir / "glove_all_results.csv"
glove_results_df.to_csv(glove_all_path, index=False)
print(f"Saved: {glove_all_path}")

In [ ]:
plt.figure(figsize=(8, 5))

for method in ["IVFPQ", "HNSWSQ", "LSH"]:
    subset = glove_results_df[glove_results_df["method"] == method]
    plt.scatter(subset["qps"], subset["recall_at_10"], label=method, alpha=0.8)

plt.xlabel("QPS")
plt.ylabel("Recall@10")
plt.title("GloVe: Recall@10 vs QPS")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
glove_recall_qps_path = plots_dir / "glove_recall_vs_qps.png"
plt.savefig(glove_recall_qps_path, dpi=150, bbox_inches="tight")
print(f"Saved: {glove_recall_qps_path}")

In [ ]:
glove_rc_values = compute_rc(xb_glove, xq_glove, k_glove, metric_glove)

In [ ]:
glove_rc_df = pd.DataFrame({
    "query_id": np.arange(len(glove_rc_values), dtype=np.int64),
    "rc": glove_rc_values,
})
glove_rc_df.head()

In [ ]:
glove_rc_stats_df = pd.DataFrame([
    {
        "mean": float(glove_rc_df["rc"].mean()),
        "std": float(glove_rc_df["rc"].std()),
        "min": float(glove_rc_df["rc"].min()),
        "max": float(glove_rc_df["rc"].max()),
    }
])
glove_rc_quantiles = glove_rc_df["rc"].quantile([0.25, 0.5, 0.75]).rename("rc_quantile")

print("GloVe RC basic stats:")
display(glove_rc_stats_df)
print("GloVe RC quantiles (0.25, 0.5, 0.75):")
display(glove_rc_quantiles)

In [ ]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
glove_rc_path = tables_dir / "glove_rc_values.csv"
glove_rc_df.to_csv(glove_rc_path, index=False)
print(f"Saved: {glove_rc_path}")

In [ ]:
glove_q25 = float(glove_rc_df["rc"].quantile(0.25))
glove_q75 = float(glove_rc_df["rc"].quantile(0.75))

glove_hard_query_idx = np.where(glove_rc_values <= glove_q25)[0]
glove_easy_query_idx = np.where(glove_rc_values >= glove_q75)[0]

In [ ]:
print(f"GloVe hard queries: {len(glove_hard_query_idx)}")
print(f"GloVe easy queries: {len(glove_easy_query_idx)}")

In [ ]:
glove_rc_split_summary_df = pd.DataFrame([
    {
        "dataset": "glove",
        "q25": glove_q25,
        "q75": glove_q75,
        "hard_query_count": int(len(glove_hard_query_idx)),
        "easy_query_count": int(len(glove_easy_query_idx)),
        "total_query_count": int(len(glove_rc_values)),
    }
])

tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
glove_split_path = tables_dir / "glove_rc_split_summary.csv"
glove_rc_split_summary_df.to_csv(glove_split_path, index=False)
print(f"Saved: {glove_split_path}")
glove_rc_split_summary_df

In [ ]:
glove_representative_configs = [
    {"method": "IVFPQ", "factory": "IVF4096,PQ20x8", "setting_name": "nprobe", "setting_value": 64},
    {"method": "HNSWSQ", "factory": "HNSW32_SQ8", "setting_name": "efSearch", "setting_value": 256},
    {"method": "LSH", "factory": "IndexLSH", "setting_name": "nbits", "setting_value": 512},
]

glove_easy_hard_rows = []
train_data_glove = xb_glove

for cfg in glove_representative_configs:
    method = cfg["method"]
    factory = cfg["factory"]
    setting_name = cfg["setting_name"]
    setting_value = cfg["setting_value"]

    if method == "LSH":
        index = faiss.IndexLSH(d_glove, int(setting_value))
    else:
        index = faiss.index_factory(d_glove, factory, metric_glove)
        if not index.is_trained:
            index.train(train_data_glove)
    index.add(xb_glove)

    if method == "IVFPQ":
        set_ivfpq_nprobe(index, int(setting_value))
    elif method == "HNSWSQ":
        set_hnswsq_efsearch(index, int(setting_value))

    D, I, total_time, qps = benchmark_index(index, xq_glove, k_glove)
    recall_all = recall_at_k(I, I_gt_glove, k_glove)
    recall_hard = recall_at_k(I[glove_hard_query_idx], I_gt_glove[glove_hard_query_idx], k_glove)
    recall_easy = recall_at_k(I[glove_easy_query_idx], I_gt_glove[glove_easy_query_idx], k_glove)

    glove_easy_hard_rows.append({
        "dataset": "glove",
        "method": method,
        "factory": factory,
        "setting_name": setting_name,
        "setting_value": setting_value,
        "recall_all": recall_all,
        "recall_hard": recall_hard,
        "recall_easy": recall_easy,
        "qps": qps,
    })

glove_easy_hard_df = pd.DataFrame(glove_easy_hard_rows)

tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
glove_easy_hard_path = tables_dir / "glove_easy_hard_results.csv"
glove_easy_hard_df.to_csv(glove_easy_hard_path, index=False)
print(f"Saved: {glove_easy_hard_path}")
glove_easy_hard_df

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8, 5))
plt.hist(glove_rc_df["rc"], bins=40, alpha=0.85, edgecolor="black")
plt.xlabel("RC")
plt.ylabel("Count")
plt.title("GloVe RC Distribution")
plt.grid(alpha=0.3)
plt.tight_layout()

glove_rc_hist_path = plots_dir / "glove_rc_histogram.png"
plt.savefig(glove_rc_hist_path, dpi=150, bbox_inches="tight")
print(f"Saved: {glove_rc_hist_path}")

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

methods = glove_easy_hard_df["method"].tolist()
x = np.arange(len(methods))
width = 0.25

plt.figure(figsize=(9, 5))
plt.bar(x - width, glove_easy_hard_df["recall_all"], width=width, label="recall_all")
plt.bar(x, glove_easy_hard_df["recall_hard"], width=width, label="recall_hard")
plt.bar(x + width, glove_easy_hard_df["recall_easy"], width=width, label="recall_easy")
plt.xticks(x, methods)
plt.ylabel("Recall@10")
plt.title("GloVe: Recall on All vs Hard vs Easy Queries")
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()

glove_easy_hard_plot_path = plots_dir / "glove_easy_vs_hard_recall.png"
plt.savefig(glove_easy_hard_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {glove_easy_hard_plot_path}")

In [ ]:
glove_sigma_values = [0.00, 0.01, 0.05, 0.10]

In [ ]:
glove_noise_rows = []
train_data_glove = xb_glove

for cfg in glove_representative_configs:
    method = cfg["method"]
    factory = cfg["factory"]
    setting_value = cfg["setting_value"]

    if method == "LSH":
        index = faiss.IndexLSH(d_glove, int(setting_value))
    else:
        index = faiss.index_factory(d_glove, factory, metric_glove)
        if not index.is_trained:
            index.train(train_data_glove)
    index.add(xb_glove)

    if method == "IVFPQ":
        set_ivfpq_nprobe(index, int(setting_value))
    elif method == "HNSWSQ":
        set_hnswsq_efsearch(index, int(setting_value))

    for sigma in glove_sigma_values:
        xq_glove_noisy = add_noise(xq_glove, sigma)
        xq_glove_noisy = l2_normalize(xq_glove_noisy)
        D, I, total_time, qps = benchmark_index(index, xq_glove_noisy, k_glove)
        recall = recall_at_k(I, I_gt_glove, k_glove)

        glove_noise_rows.append({
            "dataset": "glove",
            "method": method,
            "factory": factory,
            "noise_sigma": float(sigma),
            "recall_at_10": recall,
            "qps": qps,
        })

glove_noise_df = pd.DataFrame(glove_noise_rows)

tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
glove_noise_path = tables_dir / "glove_noisy_query_results.csv"
glove_noise_df.to_csv(glove_noise_path, index=False)
print(f"Saved: {glove_noise_path}")
glove_noise_df

In [ ]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8, 5))
for method in ["IVFPQ", "HNSWSQ", "LSH"]:
    subset = glove_noise_df[glove_noise_df["method"] == method].sort_values("noise_sigma")
    plt.plot(subset["noise_sigma"], subset["recall_at_10"], marker="o", label=method)

plt.xlabel("Noise sigma")
plt.ylabel("Recall@10")
plt.title("GloVe: Recall@10 vs Query Noise")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

glove_noise_plot_path = plots_dir / "glove_noise_vs_recall.png"
plt.savefig(glove_noise_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {glove_noise_plot_path}")

## GloVe Notes

This completes the three-dataset experimental scaffold. Next steps are aggregate cross-dataset summaries, final plots, README updates, and notebook exports.

# Cross-Dataset Summary

## Combined All-Dataset Results

In [8]:
tables_dir = PROJECT_ROOT / "results" / "tables"

sift_results_df = pd.read_csv(tables_dir / "sift_all_results.csv")
gist_results_df = pd.read_csv(tables_dir / "gist_all_results.csv")
glove_results_df = pd.read_csv(tables_dir / "glove_all_results.csv")

sift_noise_df = pd.read_csv(tables_dir / "sift_noisy_query_results.csv")
gist_noise_df = pd.read_csv(tables_dir / "gist_noisy_query_results.csv")
glove_noise_df = pd.read_csv(tables_dir / "glove_noisy_query_results.csv")

sift_easy_hard_df = pd.read_csv(tables_dir / "sift_easy_hard_results.csv")
gist_easy_hard_df = pd.read_csv(tables_dir / "gist_easy_hard_results.csv")
glove_easy_hard_df = pd.read_csv(tables_dir / "glove_easy_hard_results.csv")

In [9]:
all_results_df = pd.concat([sift_results_df, gist_results_df, glove_results_df], ignore_index=True)
all_results_sorted_df = all_results_df.sort_values(["dataset", "recall_at_10", "qps"], ascending=[True, False, False]).reset_index(drop=True)
all_results_sorted_df

,dataset,method,factory,search_param_name,search_param_value,recall_at_10,qps,total_time_sec
0,gist,HNSWSQ,HNSW32_SQ8,efSearch,256,0.94260,2115.025224,0.472808
1,gist,HNSWSQ,HNSW32_SQ8,efSearch,128,0.90150,3200.299036,0.312471
2,gist,HNSWSQ,HNSW32_SQ6,efSearch,256,0.89260,1857.099534,0.538474
3,gist,HNSWSQ,HNSW32_SQ6,efSearch,128,0.85890,2685.345933,0.372391
4,gist,HNSWSQ,HNSW32_SQ8,efSearch,64,0.82460,3928.572271,0.254545
...,...,...,...,...,...,...,...,...
103,sift,IVFPQ,"IVF4096,PQ32x8",nprobe,1,0.30939,121757.595241,0.082130
104,sift,IVFPQ,"IVF4096,PQ16x8",nprobe,1,0.28375,103121.487386,0.096973
105,sift,LSH,IndexLSH,nbits,256,0.23861,4802.314869,2.082329
106,sift,IVFPQ,"IVF4096,PQ8x8",nprobe,1,0.23670,76759.048760,0.130278


In [10]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
all_results_path = tables_dir / "all_datasets_results.csv"
all_results_df.to_csv(all_results_path, index=False)
print(f"Saved: {all_results_path}")

Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\tables\all_datasets_results.csv


## Best Configuration Per Dataset and Method

In [11]:
best_config_per_dataset_method_df = (
    all_results_df
    .sort_values(["dataset", "method", "recall_at_10", "qps"], ascending=[True, True, False, False])
    .groupby(["dataset", "method"], as_index=False)
    .first()
)
best_config_per_dataset_method_df

,dataset,method,factory,search_param_name,search_param_value,recall_at_10,qps,total_time_sec
0,gist,HNSWSQ,HNSW32_SQ8,efSearch,256,0.94260,2115.025224,0.472808
1,gist,IVFPQ,"IVF4096,PQ32x8",nprobe,64,0.27560,4892.085486,0.204412
2,gist,LSH,IndexLSH,nbits,512,0.07220,1895.365944,0.527603
3,glove,HNSWSQ,HNSW32_SQ8,efSearch,256,0.97689,9435.957168,1.059776
4,glove,IVFPQ,"IVF4096,PQ25x8",nprobe,64,0.71862,58551.128891,0.170791
5,glove,LSH,IndexLSH,nbits,512,0.60093,13864.564281,0.721263
6,sift,HNSWSQ,HNSW32_SQ8,efSearch,256,0.98687,9590.208474,1.042730
7,sift,IVFPQ,"IVF4096,PQ32x8",nprobe,64,0.71881,15495.201059,0.645361
8,sift,LSH,IndexLSH,nbits,512,0.38540,2527.956673,3.955764


In [12]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
best_cfg_path = tables_dir / "best_config_per_dataset_method.csv"
best_config_per_dataset_method_df.to_csv(best_cfg_path, index=False)
print(f"Saved: {best_cfg_path}")

Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\tables\best_config_per_dataset_method.csv


## Best Overall Configuration Per Dataset

In [13]:
best_overall_per_dataset_df = (
    all_results_df
    .sort_values(["dataset", "recall_at_10", "qps"], ascending=[True, False, False])
    .groupby("dataset", as_index=False)
    .first()
)
best_overall_per_dataset_df

,dataset,method,factory,search_param_name,search_param_value,recall_at_10,qps,total_time_sec
0,gist,HNSWSQ,HNSW32_SQ8,efSearch,256,0.94260,2115.025224,0.472808
1,glove,HNSWSQ,HNSW32_SQ8,efSearch,256,0.97689,9435.957168,1.059776
2,sift,HNSWSQ,HNSW32_SQ8,efSearch,256,0.98687,9590.208474,1.042730


In [14]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
best_overall_path = tables_dir / "best_overall_per_dataset.csv"
best_overall_per_dataset_df.to_csv(best_overall_path, index=False)
print(f"Saved: {best_overall_path}")

Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\tables\best_overall_per_dataset.csv


## High-Throughput Configuration Summary

In [15]:
high_qps_summary_df = (
    all_results_df
    .sort_values(["dataset", "method", "qps", "recall_at_10"], ascending=[True, True, False, False])
    .groupby(["dataset", "method"], as_index=False)
    .first()
)
high_qps_summary_df

,dataset,method,factory,search_param_name,search_param_value,recall_at_10,qps,total_time_sec
0,gist,HNSWSQ,HNSW32_SQ8,efSearch,16,0.59440,7705.537508,0.129777
1,gist,IVFPQ,"IVF4096,PQ8x8",nprobe,1,0.06900,19520.993846,0.051227
2,gist,LSH,IndexLSH,nbits,128,0.02300,5754.763074,0.173769
3,glove,HNSWSQ,HNSW32_SQ8,efSearch,16,0.86890,59040.520690,0.169375
4,glove,IVFPQ,"IVF4096,PQ10x8",nprobe,8,0.43570,148509.190541,0.067336
5,glove,LSH,IndexLSH,nbits,128,0.33296,42166.693367,0.237154
6,sift,HNSWSQ,HNSW32_SQ8,efSearch,16,0.85761,36972.130405,0.270474
7,sift,IVFPQ,"IVF4096,PQ32x8",nprobe,1,0.30939,121757.595241,0.082130
8,sift,LSH,IndexLSH,nbits,128,0.12899,7167.183074,1.395248


In [16]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
high_qps_path = tables_dir / "high_qps_summary.csv"
high_qps_summary_df.to_csv(high_qps_path, index=False)
print(f"Saved: {high_qps_path}")

Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\tables\high_qps_summary.csv


## Cross-Dataset Recall?QPS Plots

In [17]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

dataset_plot_specs = [
    ("sift", sift_results_df, "sift_recall_vs_qps_final.png", "SIFT: Recall@10 vs QPS"),
    ("gist", gist_results_df, "gist_recall_vs_qps_final.png", "GIST: Recall@10 vs QPS"),
    ("glove", glove_results_df, "glove_recall_vs_qps_final.png", "GloVe: Recall@10 vs QPS"),
]
style = {"IVFPQ": ("o", "tab:blue"), "HNSWSQ": ("s", "tab:orange"), "LSH": ("^", "tab:green")}

for dataset_name, df, filename, title in dataset_plot_specs:
    plt.figure(figsize=(8, 5))
    for method in ["IVFPQ", "HNSWSQ", "LSH"]:
        subset = df[df["method"] == method]
        marker, color = style[method]
        plt.scatter(subset["qps"], subset["recall_at_10"], label=method, marker=marker, color=color, alpha=0.85)
    plt.xlabel("QPS")
    plt.ylabel("Recall@10")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out_path = plots_dir / filename
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.close()

Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\plots\sift_recall_vs_qps_final.png
Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\plots\gist_recall_vs_qps_final.png
Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\plots\glove_recall_vs_qps_final.png


## Method Comparison Across Datasets

In [18]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

best_recall_method_dataset_df = (
    all_results_df
    .groupby(["dataset", "method"], as_index=False)["recall_at_10"]
    .max()
)

pivot_best = best_recall_method_dataset_df.pivot(index="dataset", columns="method", values="recall_at_10").reindex(["sift", "gist", "glove"])
datasets = pivot_best.index.tolist()
methods = [m for m in ["IVFPQ", "HNSWSQ", "LSH"] if m in pivot_best.columns]
x = np.arange(len(datasets))
width = 0.25

plt.figure(figsize=(9, 5))
for i, method in enumerate(methods):
    plt.bar(x + (i - 1) * width, pivot_best[method].values, width=width, label=method)
plt.xticks(x, datasets)
plt.ylabel("Best Recall@10")
plt.title("Best Recall@10 by Method and Dataset")
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()

best_recall_plot_path = plots_dir / "best_recall_by_method_dataset.png"
plt.savefig(best_recall_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {best_recall_plot_path}")
plt.close()

Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\plots\best_recall_by_method_dataset.png


## Robustness Summary Across Datasets

In [19]:
all_noisy_query_results_df = pd.concat([sift_noise_df, gist_noise_df, glove_noise_df], ignore_index=True)
all_noisy_query_results_df

,dataset,method,factory,noise_sigma,recall_at_10,qps
0,sift,IVFPQ,"IVF4096,PQ32x8",0.00,0.71881,9658.371799
1,sift,IVFPQ,"IVF4096,PQ32x8",0.01,0.71883,9338.737100
2,sift,IVFPQ,"IVF4096,PQ32x8",0.05,0.71885,8253.092640
3,sift,IVFPQ,"IVF4096,PQ32x8",0.10,0.71875,8777.975848
4,sift,HNSWSQ,HNSW32_SQ8,0.00,0.98685,2194.387586
5,sift,HNSWSQ,HNSW32_SQ8,0.01,0.98690,2305.682551
6,sift,HNSWSQ,HNSW32_SQ8,0.05,0.98680,5293.602517
7,sift,HNSWSQ,HNSW32_SQ8,0.10,0.98604,4560.868207
8,sift,LSH,IndexLSH,0.00,0.38540,799.026664
9,sift,LSH,IndexLSH,0.01,0.38540,1879.481553


In [20]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
all_noisy_path = tables_dir / "all_noisy_query_results.csv"
all_noisy_query_results_df.to_csv(all_noisy_path, index=False)
print(f"Saved: {all_noisy_path}")

Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\tables\all_noisy_query_results.csv


In [21]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
dataset_order = ["sift", "gist", "glove"]
for ax, dataset_name in zip(axes, dataset_order):
    ds_df = all_noisy_query_results_df[all_noisy_query_results_df["dataset"] == dataset_name]
    for method in ["IVFPQ", "HNSWSQ", "LSH"]:
        subset = ds_df[ds_df["method"] == method].sort_values("noise_sigma")
        ax.plot(subset["noise_sigma"], subset["recall_at_10"], marker="o", label=method)
    ax.set_title(dataset_name.upper())
    ax.set_xlabel("Noise sigma")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Recall@10")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
fig.suptitle("Noise Robustness Comparison Across Datasets", y=1.03)
fig.tight_layout()

noise_cmp_path = plots_dir / "noise_robustness_comparison.png"
fig.savefig(noise_cmp_path, dpi=150, bbox_inches="tight")
print(f"Saved: {noise_cmp_path}")
plt.close(fig)

Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\plots\noise_robustness_comparison.png


## Easy vs Hard Summary Across Datasets

In [22]:
all_easy_hard_results_df = pd.concat([sift_easy_hard_df, gist_easy_hard_df, glove_easy_hard_df], ignore_index=True)
all_easy_hard_results_df

,dataset,method,factory,setting_name,setting_value,recall_all,recall_hard,recall_easy,qps
0,sift,IVFPQ,"IVF4096,PQ32x8",nprobe,64,0.71881,0.66272,0.77900,12829.004635
1,sift,HNSWSQ,HNSW32_SQ8,efSearch,256,0.98686,0.98428,0.98764,7389.574994
2,sift,LSH,IndexLSH,nbits,512,0.38540,0.34580,0.43832,1034.093013
3,gist,IVFPQ,"IVF4096,PQ32x8",nprobe,64,0.27560,0.21720,0.34080,4647.742918
4,gist,HNSWSQ,HNSW32_SQ8,efSearch,256,0.94310,0.90440,0.96560,1136.429110
5,gist,LSH,IndexLSH,nbits,512,0.07220,0.05560,0.10000,1317.874316
6,glove,IVFPQ,"IVF4096,PQ20x8",nprobe,64,0.65636,0.55392,0.72548,52768.499324
7,glove,HNSWSQ,HNSW32_SQ8,efSearch,256,0.97677,0.95580,0.98600,10109.301771
8,glove,LSH,IndexLSH,nbits,512,0.60093,0.47508,0.70088,14942.840646


In [23]:
tables_dir = PROJECT_ROOT / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
all_easy_hard_path = tables_dir / "all_easy_hard_results.csv"
all_easy_hard_results_df.to_csv(all_easy_hard_path, index=False)
print(f"Saved: {all_easy_hard_path}")

Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\tables\all_easy_hard_results.csv


In [24]:
plots_dir = PROJECT_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

eh_plot_df = all_easy_hard_results_df.copy()
eh_plot_df["label"] = eh_plot_df["dataset"].str.upper() + "-" + eh_plot_df["method"]
x = np.arange(len(eh_plot_df))
width = 0.25

plt.figure(figsize=(14, 5))
plt.bar(x - width, eh_plot_df["recall_all"], width=width, label="recall_all")
plt.bar(x, eh_plot_df["recall_hard"], width=width, label="recall_hard")
plt.bar(x + width, eh_plot_df["recall_easy"], width=width, label="recall_easy")
plt.xticks(x, eh_plot_df["label"], rotation=30, ha="right")
plt.ylabel("Recall@10")
plt.title("Easy/Hard Recall Comparison Across Datasets and Methods")
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()

easy_hard_cmp_path = plots_dir / "easy_hard_comparison.png"
plt.savefig(easy_hard_cmp_path, dpi=150, bbox_inches="tight")
print(f"Saved: {easy_hard_cmp_path}")
plt.close()

Saved: C:\Users\vaspn\Downloads\Vector-Search-FAISS\results\plots\easy_hard_comparison.png


## Final Experimental Observations

- HNSWSQ tends to dominate recall across the tested settings.
- IVFPQ tends to dominate throughput in many settings.
- LSH is generally weaker in recall than IVFPQ and HNSWSQ.
- Increasing search parameters typically improves recall while reducing QPS.
- RC and noisy-query analysis capture query hardness and robustness trends.
- Next steps: update README, export the notebook, and prepare the poster/report figures.